# Bộ đánh giá RAG end-to-end

Bạn dựng một **harness** đánh giá RAG như unit test cho chất lượng: cùng một tập câu hỏi ghim, nhiều cấu hình
(đơn vị chunk, hybrid, rerank), mỗi cấu hình ra một dòng gồm thước đo **retrieval** (recall@5, MRR@10), thước đo
**generation** do LLM chấm (faithfulness, answer relevance), tỉ lệ từ chối và độ trễ. Rồi bạn kiểm chính giám khảo
trước khi tin nó, và viết `REPORT.md` khuyến nghị một cấu hình.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp, hằng. Test gọi `recall_at_k`, `mrr_at_k`, `judge_faithfulness`,
  `judge_relevance`, `run_eval` với LLM giả; giữ nguyên tên, tham số và các hằng `FAITHFULNESS_SCHEMA`,
  `RELEVANCE_SCHEMA`, `COLUMNS`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, gọi Ollama, in bảng; test bỏ qua chúng.
- Cần Ollama với `qwen2.5:3b`, `bge-m3` (và `qwen2.5:7b` nếu làm judge), cùng cache của ba bài Intermediate trong
  `data/raw/zalo-legal/cache/`: `bge-m3.npy` (bài 01), `hybrid-top100.json` (bài 01),
  `rerank-bge-reranker-v2-m3-top50.json` (bài 03).
- Chấm: `uv run pytest tracks/ai-engineer/modules/13-rag/exercises/advanced/01-rag-eval-harness -q`

In [ ]:
# explore: tìm gốc repo, tải Zalo Legal nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

ZALO = ensure("zalo-legal")
CACHE = ZALO / "cache"
print(ZALO, sorted(p.name for p in CACHE.iterdir()) if CACHE.exists() else "chưa có cache: làm các bài Intermediate trước")

In [ ]:
import json
import math
import re
import statistics
import time
from dataclasses import dataclass
from typing import Callable

import numpy as np
import pandas as pd

import llmkit
from llmkit import LLMClient

## 1. Thước đo retrieval (dán từ bài Basic 03)

Notebook tự chứa: **dán** `recall_at_k` và `mrr_at_k` của bạn ở bài Basic 03 vào cell dưới, giữ nguyên chữ ký
(`retrieved[i]` là danh sách uid theo hạng của câu i, `relevant[i]` là tập uid đúng). Đừng import chéo bài. Test kiểm
lại chúng bằng ví dụ tính tay ở mục 6 lý thuyết.

In [ ]:
def recall_at_k(retrieved: list[list[str]], relevant: list[set[str]], k: int) -> float:
    """Trung bình trên các truy vấn của |Rel ∩ top-k| / |Rel| (Basic 03)."""
    # TODO: dán lời giải bài Basic 03
    raise NotImplementedError


def mrr_at_k(retrieved: list[list[str]], relevant: list[set[str]], k: int) -> float:
    """Trung bình 1 / hạng tài liệu đúng đầu tiên trong top-k, không có thì 0 (Basic 03)."""
    # TODO: dán lời giải bài Basic 03
    raise NotImplementedError

## 2. Hợp đồng dữ liệu của harness

Harness không biết pipeline bên trong. Mỗi **cấu hình** là một hàm `config(question_text) -> RagOutput`:

- `retrieved`: uid các **điều** theo hạng retrieval (ít nhất 10 phần tử để tính MRR@10).
- `context`: các cặp `(uid, đoạn văn)` **thật sự** đưa vào prompt, theo thứ tự. Judge faithfulness đọc đúng các đoạn này.
- `answer`, `refused`: câu trả lời và cờ từ chối (như `RagAnswer` của bài Intermediate 02).
- `seconds`: thời gian cấu hình tự đo (retrieval, rerank, sinh). `None` thì `run_eval` tự đo thời gian gọi hàm. Cần
  trường này vì khi câu trả lời đã cache, thời gian gọi hàm không còn là độ trễ thật.

Hai schema JSON và danh sách cột dưới đây là **cố định** (test trả JSON đúng theo schema này): bạn viết prompt và phần
đọc kết quả, không đổi schema.

In [ ]:
@dataclass
class Question:
    qid: str
    text: str
    relevant: set[str]          # uid các điều đúng (qrels)


@dataclass
class RagOutput:
    retrieved: list[str]                 # uid điều theo hạng retrieval
    context: list[tuple[str, str]]       # (uid, đoạn văn) đưa vào prompt, theo thứ tự
    answer: str
    refused: bool
    seconds: float | None = None         # độ trễ do cấu hình tự đo; None thì run_eval tự đo


# Judge tách câu trả lời thành các khẳng định, đánh dấu từng khẳng định có được nguồn ủng hộ không.
FAITHFULNESS_SCHEMA = {
    "type": "object",
    "properties": {
        "claims": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {"claim": {"type": "string"}, "supported": {"type": "boolean"}},
                "required": ["claim", "supported"],
            },
        }
    },
    "required": ["claims"],
}

# Judge viết lý do ngắn trước rồi cho điểm 1..5 (lý do trước điểm giúp model nhỏ chấm ổn định hơn).
RELEVANCE_SCHEMA = {
    "type": "object",
    "properties": {"reason": {"type": "string"}, "score": {"type": "integer", "minimum": 1, "maximum": 5}},
    "required": ["reason", "score"],
}

COLUMNS = ["config", "n", "recall@5", "mrr@10", "context_hit", "refused", "faithfulness", "relevance",
           "judge_nan", "latency_s"]

## 3. LLM-as-judge

`judge_faithfulness(answer, passages, llm) -> float`:

- Gọi `llm.chat(messages, json_schema=FAITHFULNESS_SCHEMA)` **một lần**. Prompt chứa **mọi** đoạn trong `passages` và
  câu trả lời; dặn judge tách câu trả lời thành khẳng định, `supported = true` **chỉ khi** nguồn nói rõ hoặc suy ra trực
  tiếp (đúng theo hiểu biết chung nhưng không có trong nguồn vẫn là `false`, như định nghĩa ở mục 6 lý thuyết).
- Trả tỉ lệ khẳng định có `supported = true`, trong [0, 1].
- **Không bao giờ crash vì judge**: JSON hỏng, thiếu khoá, sai kiểu (`supported` không phải bool), danh sách khẳng định
  rỗng thì trả `math.nan`. Model nhỏ thỉnh thoảng trả JSON sai; một câu hỏng không được làm mất 40 phút chấm.

`judge_relevance(question, answer, llm) -> float`:

- Gọi `llm.chat(messages, json_schema=RELEVANCE_SCHEMA)` một lần, thang 1..5 có mô tả từng mức (5 = trả lời trực tiếp,
  đầy đủ; 1 = không trả lời câu hỏi). Chỉ chấm câu trả lời có giải quyết **câu hỏi được hỏi** không, không chấm đúng sai
  pháp lý (đó là việc của faithfulness và retrieval).
- Trả `(score - 1) / 4` trong [0, 1]; JSON hỏng, thiếu `score`, điểm không phải số nguyên trong 1..5 thì `math.nan`.

`llm` là bất kỳ client nào có `chat(messages, *, json_schema=...)`: client của `llmkit` khi chạy thật, `FakeLLM` trong
test.

In [ ]:
def judge_faithfulness(answer: str, passages: list[str], llm: LLMClient) -> float:
    """Tỉ lệ khẳng định trong answer được passages ủng hộ, theo judge; JSON hỏng hoặc không có khẳng định thì nan."""
    # TODO
    raise NotImplementedError


def judge_relevance(question: str, answer: str, llm: LLMClient) -> float:
    """(điểm 1..5 - 1) / 4 cho mức answer trả lời đúng question; JSON hỏng hoặc điểm sai thì nan."""
    # TODO
    raise NotImplementedError

## 4. `run_eval`

`run_eval(configs, questions, *, judge=None) -> pd.DataFrame`, `judge=None` nghĩa là `llmkit.get_client()`:

- Với mỗi cấu hình theo thứ tự của `configs`, gọi `config(q.text)` **đúng một lần** cho mỗi câu hỏi.
- Một dòng mỗi cấu hình, cột đúng `COLUMNS` theo thứ tự:

| Cột | Giá trị |
| --- | --- |
| `config` | tên cấu hình (khoá của `configs`) |
| `n` | số câu hỏi |
| `recall@5`, `mrr@10` | `recall_at_k(..., 5)`, `mrr_at_k(..., 10)` trên `retrieved` |
| `context_hit` | tỉ lệ câu có ít nhất một uid trong `context` thuộc `relevant` (điều đúng có tới được LLM không) |
| `refused` | tỉ lệ câu `refused = True` |
| `faithfulness`, `relevance` | trung bình điểm judge trên các câu **không từ chối**, **bỏ qua `nan`**; không còn điểm nào thì `nan` |
| `judge_nan` | số lần judge trả `nan` (cả hai judge cộng lại), để biết điểm trung bình dựa trên bao nhiêu câu |
| `latency_s` | **trung vị** độ trễ mỗi câu: `seconds` của `RagOutput`, hoặc thời gian gọi `config` nếu `seconds` là `None` |

- Câu bị từ chối **không** được gửi cho judge: câu từ chối không có khẳng định nào để kiểm, và tốn lượt gọi vô ích.
  Tỉ lệ từ chối đã có cột riêng.
- Gọi judge qua hai hàm ở mục 3 (`judge_faithfulness(answer, [đoạn văn của context], judge)` và
  `judge_relevance(q.text, answer, judge)`).

In [ ]:
def run_eval(configs: dict[str, Callable[[str], RagOutput]], questions: list[Question], *,
             judge: LLMClient | None = None) -> pd.DataFrame:
    """Một dòng mỗi cấu hình, cột đúng COLUMNS; judge chỉ chấm câu không từ chối, trung bình bỏ qua nan."""
    # TODO
    raise NotImplementedError

## 5. Chạy thật trên 50 câu ghim

Các cell dưới dựng 4 cấu hình trên **50 câu đầu** của tập ghim (300 câu đầu `qrels/test.jsonl`, như các bài
Intermediate), sinh câu trả lời bằng `qwen2.5:3b` theo đúng prompt của bài Intermediate 02:

| Cấu hình | Retrieval (thứ hạng điều) | Nguồn đưa vào prompt |
| --- | --- | --- |
| `dense · cả điều` | dense `bge-m3` trên toàn corpus | 5 điều đầu, mỗi điều cắt ở 1 500 ký tự |
| `dense · khoản` | dense `bge-m3` | 5 khoản gần câu hỏi nhất trong 5 điều đầu |
| `hybrid · khoản` | top-100 hybrid RRF của bài 01 | như trên |
| `hybrid + rerank · khoản` | top-50 hybrid xếp lại bằng điểm cross-encoder đã cache ở bài 03 | như trên |

Hai cấu hình đầu đã viết sẵn làm mẫu; bạn viết hai cấu hình sau (`# TODO` trong cell). Câu trả lời được **cache ra
đĩa** theo (cấu hình, câu hỏi): sinh một lần (khoảng 200 lượt gọi `qwen2.5:3b`, 10 phút trên laptop GPU 4GB), sau đó
đổi judge hay prompt judge thì chạy lại `run_eval` không phải sinh lại.

Độ trễ của cấu hình rerank cộng thêm thời gian cross-encoder **đã đo** ở bài 03 (khoảng 35 giây một câu trên CPU):
cache điểm giúp chạy lại nhanh nhưng không được làm độ trễ báo cáo nhỏ đi.

In [ ]:
# explore: corpus, 50 câu đầu của tập ghim, cache của các bài Intermediate; embed câu hỏi một lần
N_EVAL = 50

corpus: dict[str, tuple[str, str]] = {}
with open(ZALO / "corpus.jsonl", encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        corpus[d["_id"]] = (d["title"], d["text"])
qrels: dict[str, set[str]] = {}
with open(ZALO / "qrels" / "test.jsonl", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        qrels.setdefault(row["query-id"], set()).add(row["corpus-id"])
with open(ZALO / "queries.jsonl", encoding="utf-8") as f:
    query_text = {q["_id"]: q["text"] for q in map(json.loads, f)}
questions = [Question(qid, query_text[qid], rel) for qid, rel in list(qrels.items())[:N_EVAL]]
qid_of = {q.text: q.qid for q in questions}

for name in ["bge-m3.npy", "bge-m3.ids.json", "hybrid-top100.json", "rerank-bge-reranker-v2-m3-top50.json"]:
    if not (CACHE / name).exists():
        raise RuntimeError(f"Thiếu cache {name}: làm bài Intermediate 01 (bge-m3, hybrid) và 03 (rerank) trước.")
uids = json.loads((CACHE / "bge-m3.ids.json").read_text(encoding="utf-8"))
row_of = {u: i for i, u in enumerate(uids)}
matrix = np.load(CACHE / "bge-m3.npy")
hybrid = json.loads((CACHE / "hybrid-top100.json").read_text(encoding="utf-8"))
rerank_cache = json.loads((CACHE / "rerank-bge-reranker-v2-m3-top50.json").read_text(encoding="utf-8"))


def unit(v: np.ndarray) -> np.ndarray:
    return v / np.linalg.norm(v, axis=-1, keepdims=True)


t0 = time.perf_counter()
QVEC: dict[str, np.ndarray] = {}
EMBED_S: dict[str, float] = {}
for q in questions:
    t1 = time.perf_counter()
    QVEC[q.text] = unit(llmkit.embed([q.text])[0])
    EMBED_S[q.text] = time.perf_counter() - t1
print(f"{len(corpus)} điều, {len(questions)} câu, embed câu hỏi {time.perf_counter() - t0:.1f} s")

In [ ]:
# explore: dán từ bài Intermediate 02: Article, REFUSAL, build_messages (cùng prompt thì so cấu hình mới công bằng)
REFUSAL = "Không tìm thấy căn cứ trong tài liệu được cung cấp."


@dataclass
class Article:
    uid: str
    title: str
    text: str


def build_messages(question: str, passages: list[Article]) -> list[dict]:
    # TODO: dán lời giải bài Intermediate 02
    raise NotImplementedError


KHOAN = re.compile(r"\d+\.\s")


def split_khoan(a: Article) -> list[Article]:
    """Một Article mỗi khoản (như bài Intermediate 02), mỗi khoản tối đa 2 000 ký tự."""
    groups: list[list[str]] = []
    for line in a.text.split("\n"):
        if not line.strip():
            continue
        if KHOAN.match(line.strip()) or not groups:
            groups.append([line.strip()])
        else:
            groups[-1].append(line.strip())
    return [Article(a.uid, a.title, "\n".join(g)[:2000]) for g in groups] or [a]

In [ ]:
# explore: 4 cấu hình = (thứ hạng điều, cách chọn nguồn, độ trễ cộng thêm)
GEN_MODEL = "qwen2.5:3b"
MIN_SCORE = 0.5                                   # cổng từ chối trên cosine (như bài Intermediate 02)


def article(uid: str) -> Article:
    return Article(uid, *corpus[uid])


def rank_dense(text: str) -> list[str]:
    return [uids[i] for i in np.argsort(-(matrix @ QVEC[text]), kind="stable")[:100]]


def rank_hybrid(text: str) -> list[str]:
    return hybrid[qid_of[text]]


def rank_rerank(text: str) -> list[str]:
    """Top-50 hybrid xếp lại theo điểm cross-encoder đã cache ở bài 03 (bằng điểm giữ thứ tự cũ), rồi phần còn lại."""
    # TODO
    raise NotImplementedError


def whole_articles(text: str, ranking: list[str]) -> list[tuple[Article, float]]:
    """5 điều đầu, mỗi điều cắt ở 1 500 ký tự; điểm = cosine của điều với câu hỏi."""
    return [(Article(u, corpus[u][0], corpus[u][1][:1500]), float(matrix[row_of[u]] @ QVEC[text]))
            for u in ranking[:5]]


def best_khoan(text: str, ranking: list[str]) -> list[tuple[Article, float]]:
    """Tách 5 điều đầu theo khoản, giữ 5 khoản có cosine cao nhất với câu hỏi."""
    chunks = [c for u in ranking[:5] for c in split_khoan(article(u))]
    scores = unit(llmkit.embed([c.title + "\n" + c.text for c in chunks])) @ QVEC[text]
    return [(chunks[i], float(scores[i])) for i in np.argsort(-scores, kind="stable")[:5]]


def no_extra(text: str) -> float:
    return 0.0


PIPELINES = {
    "dense · cả điều": ("dense-article", rank_dense, whole_articles, no_extra),
    "dense · khoản": ("dense-khoan", rank_dense, best_khoan, no_extra),
    # TODO: "hybrid · khoản" (rank_hybrid, best_khoan) và "hybrid + rerank · khoản" (rank_rerank, best_khoan, cộng
    # thời gian rerank đã đo ở bài 03: rerank_cache[qid]["seconds"])
}

In [ ]:
# explore: hai pha, cache ra đĩa. GPU 4GB chỉ giữ được MỘT model: xen kẽ embed (bge-m3) và sinh (qwen2.5:3b) làm
# Ollama nạp lại model mỗi lượt, chậm gấp đôi và độ trễ đo được lẫn cả thời gian nạp. Nên retrieve hết rồi mới sinh.
answers_path = CACHE / f"adv01-answers-{GEN_MODEL.replace(':', '-')}.json"
saved = json.loads(answers_path.read_text(encoding="utf-8")) if answers_path.exists() else {}
todo = [(key, rank_fn, context_fn, extra, q) for key, rank_fn, context_fn, extra in PIPELINES.values()
        for q in questions if f"{key}|{q.qid}" not in saved]

t0 = time.perf_counter()
prepared = {}
for key, rank_fn, context_fn, extra, q in todo:                       # pha 1: retrieval (bge-m3)
    t1 = time.perf_counter()
    ranking = rank_fn(q.text)
    hits = context_fn(q.text, ranking)
    prepared[f"{key}|{q.qid}"] = (ranking, hits, EMBED_S[q.text] + time.perf_counter() - t1 + extra(q.text))
print(f"pha 1 (retrieval): {len(prepared)} lượt, {time.perf_counter() - t0:.0f} s", flush=True)

t0 = time.perf_counter()
for n, (k, (ranking, hits, seconds)) in enumerate(prepared.items(), start=1):   # pha 2: sinh (qwen2.5:3b)
    q_text = query_text[k.split("|")[1]]
    t1 = time.perf_counter()
    if not hits or max(s for _, s in hits) < MIN_SCORE:
        reply = REFUSAL
    else:
        reply = llmkit.chat(build_messages(q_text, [a for a, _ in hits]), model=GEN_MODEL)
    saved[k] = {
        "retrieved": ranking[:100],
        "context": [[a.uid, f"{a.uid.rsplit('+', 1)[0]}, {a.title}\n{a.text}"] for a, _ in hits],
        "answer": reply,
        "refused": "Không tìm thấy căn cứ" in reply,
        "seconds": seconds + time.perf_counter() - t1,
    }
    if n % 20 == 0:
        print(f"pha 2 (sinh): {n}/{len(prepared)}", flush=True)
        _ = answers_path.write_text(json.dumps(saved, ensure_ascii=False), encoding="utf-8")
_ = answers_path.write_text(json.dumps(saved, ensure_ascii=False), encoding="utf-8")
print(f"pha 2 (sinh): {time.perf_counter() - t0:.0f} s")


def cached_config(key: str) -> Callable[[str], RagOutput]:
    """Cấu hình cho run_eval: đọc câu trả lời đã sinh; seconds là độ trễ đo lúc sinh."""
    def config(text: str) -> RagOutput:
        d = saved[f"{key}|{qid_of[text]}"]
        return RagOutput(d["retrieved"], [tuple(c) for c in d["context"]], d["answer"], d["refused"], d["seconds"])
    return config


CONFIGS = {label: cached_config(spec[0]) for label, spec in PIPELINES.items()}
print(CONFIGS["dense · khoản"](questions[0].text).answer[:300])

In [ ]:
# explore: chạy harness, khoảng 300 lượt gọi judge. Judge nên mạnh hơn (hoặc khác) model sinh: qwen2.5:7b trên PC
# 12GB; trên laptop GPU 4GB 7b vẫn chạy nhưng chậm (ước khoảng 1 giờ vì không vừa VRAM); hoặc dùng qwen2.5:3b (ước
# 25 phút) và ghi rõ giới hạn trong REPORT.md.
JUDGE_MODEL = "qwen2.5:7b"


class WithModel:
    """Client của llmkit nhưng gọi chat bằng một model cố định (judge khác model sinh)."""

    def __init__(self, model: str):
        self.model = model

    def chat(self, messages, **kwargs):
        return llmkit.chat(messages, model=self.model, **kwargs)

    def embed(self, texts, **kwargs):
        return llmkit.embed(texts, **kwargs)


judge = WithModel(JUDGE_MODEL)
# Tuỳ chọn: judge mạnh hơn qua một gateway tương thích OpenAI (OmniRoute, Gemini...). Khai báo trong setup/.env
# (không commit key): OMNIROUTE_BASE_URL (kết thúc bằng /v1), OMNIROUTE_API_KEY, OMNIROUTE_MODEL. Bỏ comment 4 dòng dưới.
# import os
# from testkit import _load_env_file; _load_env_file(ROOT / "setup" / ".env")
# judge = llmkit.OpenAICompatClient(base_url=os.environ["OMNIROUTE_BASE_URL"], api_key=os.environ["OMNIROUTE_API_KEY"],
#                                   model=os.environ["OMNIROUTE_MODEL"]); JUDGE_MODEL = judge.model
t0 = time.perf_counter()
table = run_eval(CONFIGS, questions, judge=judge)
print(f"judge {JUDGE_MODEL}: {(time.perf_counter() - t0) / 60:.1f} phút")
display(table.round(3))

In [ ]:
# explore: kiểm judge trước khi tin: chấm cùng câu trả lời với nguồn ĐÚNG và với nguồn của câu hỏi KẾ TIẾP
# (câu trả lời chắc chắn không được nguồn đó ủng hộ). Judge tốt: nguồn sai cho điểm gần 0.
name = "dense-khoan"
answered = [q for q in questions if not saved[f"{name}|{q.qid}"]["refused"]]
pairs = []
for i, q in enumerate(answered):
    mine, other = saved[f"{name}|{q.qid}"], saved[f"{name}|{answered[(i + 1) % len(answered)].qid}"]
    right = judge_faithfulness(mine["answer"], [t for _, t in mine["context"]], judge)
    wrong = judge_faithfulness(mine["answer"], [t for _, t in other["context"]], judge)
    pairs.append((right, wrong))
pairs = np.array(pairs, dtype=float)
print(f"{len(pairs)} câu trả lời | nguồn đúng: trung bình {np.nanmean(pairs[:, 0]):.2f} | nguồn sai: trung bình "
      f"{np.nanmean(pairs[:, 1]):.2f}, chấm >= 0.5: {(pairs[:, 1] >= 0.5).sum()}, chấm 1.0: {(pairs[:, 1] == 1).sum()}")

In [ ]:
# explore: hiệu chỉnh bằng người chấm. Lần đầu: ghi 20 câu trả lời ra calibration.csv; bạn tự đọc nguồn và điền cột
# `human` (1 = mọi ý đều có trong nguồn, 0 = có ý không có trong nguồn). Chạy lại cell để đo đồng ý với judge.
calib_path = Path("calibration.csv")
if not calib_path.exists():
    rows = []
    for q in answered[:20]:
        d = saved[f"{name}|{q.qid}"]
        rows.append({"qid": q.qid, "question": q.text, "answer": d["answer"],
                     "sources": "\n\n".join(t for _, t in d["context"]),
                     "judge": judge_faithfulness(d["answer"], [t for _, t in d["context"]], judge), "human": ""})
    pd.DataFrame(rows).to_csv(calib_path, index=False)
    print(f"đã ghi {calib_path.resolve()}: điền cột human rồi chạy lại cell này")
else:
    calib = pd.read_csv(calib_path).dropna(subset=["human", "judge"])
    judge_says = (calib["judge"] == 1.0).astype(int)
    agree = (judge_says == calib["human"].astype(int)).mean()
    print(f"{len(calib)} câu có nhãn | đồng ý judge/người: {agree:.0%}")
    display(pd.crosstab(calib["human"].astype(int), judge_says, rownames=["người"], colnames=["judge = 1.0"]))

## Đọc kết quả và viết `REPORT.md`

Trước khi viết, tự trả lời:

- Cấu hình nào có `recall@5` cao nhất, cấu hình nào có `context_hit` cao nhất? Vì sao hai cột này khác nhau ở cấu hình
  cắt cả điều và cấu hình chọn khoản, dù dùng cùng thứ hạng dense?
- Faithfulness và tỉ lệ từ chối đổi thế nào khi nguồn là khoản thay cho cả điều bị cắt cụt? Cột nào đổi nhiều hơn?
- Rerank cải thiện `mrr@10` bao nhiêu, và đổi lại độ trễ bao nhiêu lần? Mức cải thiện có lớn hơn nhiễu của 50 câu
  (mỗi câu đóng góp 0.02) không?
- Kiểm judge: nguồn sai có bị chấm thấp không? Judge và bạn đồng ý bao nhiêu trên 20 câu? Nếu đồng ý dưới 80%, các
  cột faithfulness nói được gì?

Rồi điền `REPORT.md` (cùng thư mục) và làm phần **trắc nghiệm** của bài trong app.